## Simulation work for SGIMC

In [ ]:
import numpy as np


def get_metrics(real_score, predict_score):
    sorted_predict_score = np.array(
        sorted(list(set(np.array(predict_score).flatten())))
    )
    sorted_predict_score_num = len(sorted_predict_score)
    thresholds = sorted_predict_score[
        np.int32(sorted_predict_score_num * np.arange(1, 1000) / 1000)
    ]
    thresholds = np.mat(thresholds)
    thresholds_num = thresholds.shape[1]

    predict_score_matrix = np.tile(predict_score, (thresholds_num, 1))
    negative_index = np.where(predict_score_matrix < thresholds.T)
    positive_index = np.where(predict_score_matrix >= thresholds.T)
    predict_score_matrix[negative_index] = 0
    predict_score_matrix[positive_index] = 1
    TP = predict_score_matrix.dot(real_score.T)
    FP = predict_score_matrix.sum(axis=1) - TP
    FN = real_score.sum() - TP
    TN = len(real_score.T) - TP - FP - FN

    fpr = FP / (FP + TN)
    tpr = TP / (TP + FN)
    ROC_dot_matrix = np.mat(sorted(np.column_stack((fpr, tpr)).tolist())).T
    ROC_dot_matrix.T[0] = [0, 0]
    ROC_dot_matrix = np.c_[ROC_dot_matrix, [1, 1]]
    x_ROC = ROC_dot_matrix[0].T
    y_ROC = ROC_dot_matrix[1].T
    auc = 0.5 * (x_ROC[1:] - x_ROC[:-1]).T * (y_ROC[:-1] + y_ROC[1:])

    recall_list = tpr
    precision_list = TP / (TP + FP)
    PR_dot_matrix = np.mat(
        sorted(np.column_stack((recall_list, precision_list)).tolist())
    ).T
    PR_dot_matrix.T[0] = [0, 1]
    PR_dot_matrix = np.c_[PR_dot_matrix, [1, 0]]
    x_PR = PR_dot_matrix[0].T
    y_PR = PR_dot_matrix[1].T
    aupr = 0.5 * (x_PR[1:] - x_PR[:-1]).T * (y_PR[:-1] + y_PR[1:])

    f1_score_list = 2 * TP / (len(real_score.T) + TP - TN)
    accuracy_list = (TP + TN) / len(real_score.T)
    specificity_list = TN / (TN + FP)

    max_index = np.argmax(f1_score_list)
    f1_score = f1_score_list[max_index]
    accuracy = accuracy_list[max_index]
    specificity = specificity_list[max_index]
    recall = recall_list[max_index]
    precision = precision_list[max_index]
    return [aupr[0, 0], auc[0, 0], f1_score, accuracy, recall, specificity, precision]

In [ ]:
import os
import gzip
import pickle
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.model_selection import ParameterGrid, KFold, train_test_split

from sgimc import SparseGroupIMCClassifier
from sgimc.utils import mc_split, get_submatrix


# ====== paths ======
PATH_TO_EXP = "/Users/sijianfan/projects/BiSSGL/datasets/simulations"
PATH_DATA = os.path.join(PATH_TO_EXP, "n_features")
PATH_OUTPUT = os.path.join(PATH_TO_EXP, "outputs_sgimc")
os.makedirs(PATH_OUTPUT, exist_ok=True)

# ====== simulation dataset grid ======
n_features_grid = np.arange(50, 501, 50)
n_repeats = 50
filename_template = "data_feature_{:03d}_rep_{:02d}.gz"

dataset_grid = []
for feature_id, n_features in enumerate(n_features_grid):
    for repeat_id in range(n_repeats):
        dataset_grid.append(
            {
                "feature_id": feature_id,
                "n_features": int(n_features),
                "repeat_id": int(repeat_id),
                "filename": os.path.join(
                    PATH_DATA,
                    filename_template.format(n_features, repeat_id),
                ),
            }
        )

print("Number of datasets:", len(dataset_grid))

Number of datasets: 500


In [10]:
# ====== choose one feature size for this notebook ======
target_n_features = 50
n_repeats = 50
filename_template = "data_feature_{:03d}_rep_{:02d}.gz"

dataset_grid = []
for repeat_id in range(n_repeats):
    dataset_grid.append(
        {
            "n_features": int(target_n_features),
            "repeat_id": int(repeat_id),
            "filename": os.path.join(
                PATH_DATA,
                filename_template.format(target_n_features, repeat_id),
            ),
        }
    )

print("Number of datasets:", len(dataset_grid))

Number of datasets: 50


In [ ]:
# ====== parameter grids ======
grid_dataset = ParameterGrid(
    {
        "train_size": np.arange(0.05, 0.51, 0.05),
        "n_splits": [3],
    }
)

grid_model = ParameterGrid(
    {
        "C_lasso": [1.0, 1e-2, 1e-4],
        "C_group": [1.0, 1e-2, 1e-4],
        "C_ridge": [1.0, 1e-2, 1e-4],
        "rank": [25],
    }
)


# ====== random seed ======
random_state = 42
dvlp_size, test_size = 0.9, 0.1


# ====== main loop ======
results = []

for ds in tqdm(dataset_grid, desc="Datasets"):
    # load one dataset
    with gzip.open(ds["filename"], "rb") as fin:
        data = pickle.load(fin)

    U = data["X"]
    V = data["Y"]
    Y = data["R_noisy"].copy()
    Y_true = data["R"].copy()

    # split development / test
    ind_dvlp, ind_test = next(
        mc_split(
            Y,
            n_splits=1,
            random_state=random_state,
            train_size=dvlp_size,
            test_size=test_size,
        )
    )

    Y_test = get_submatrix(Y_true, ind_test)

    # loop over training settings
    for par_dtst in grid_dataset:
        # prepare the train dataset: take the specified share from the beginning of the index array
        ind_train_all, _ = train_test_split(
            ind_dvlp,
            shuffle=False,
            random_state=random_state,
            test_size=(1 - (par_dtst["train_size"] / dvlp_size)),
        )

        # loop over model settings
        for par_mdl in grid_model:
            C_lasso, C_group, C_ridge = (
                par_mdl["C_lasso"],
                par_mdl["C_group"],
                par_mdl["C_ridge"],
            )

            imc = SparseGroupIMCClassifier(
                par_mdl["rank"],
                n_threads=-1,
                random_state=42,
                C_lasso=C_lasso,
                C_group=C_group,
                C_ridge=C_ridge,
            )

            # fit on the whole development dataset
            Y_train = get_submatrix(Y, ind_train_all)
            Y_train[Y_train == 0] = -1

            imc.fit(U, V, Y_train)

            # get test score
            prob_full = imc.predict_proba(U, V)
            prob_test = get_submatrix(prob_full, ind_test)
            scores_test = get_metrics((Y_test.data + 1) / 2, prob_test.data)

            d1_test = int(sum(abs(imc.coef_W_).max(axis=1) > 0))
            d2_test = int(sum(abs(imc.coef_H_).max(axis=1) > 0))

            # k-fold CV
            splt = KFold(
                par_dtst["n_splits"],
                shuffle=True,
                random_state=random_state,
            )

            for cv, (ind_train, ind_valid) in enumerate(splt.split(ind_train_all)):
                ind_train = ind_train_all[ind_train]
                ind_valid = ind_train_all[ind_valid]

                Y_train = get_submatrix(Y, ind_train)
                Y_valid = get_submatrix(Y, ind_valid)

                Y_train[Y_train == 0] = -1

                imc = SparseGroupIMCClassifier(
                    par_mdl["rank"],
                    n_threads=-1,
                    random_state=42,
                    C_lasso=C_lasso,
                    C_group=C_group,
                    C_ridge=C_ridge,
                )
                imc.fit(U, V, Y_train)

                # validation score
                prob_full = imc.predict_proba(U, V)
                prob_valid = get_submatrix(prob_full, ind_valid)
                scores_valid = get_metrics((Y_valid.data + 1) / 2, prob_valid.data)

                d1_valid = int(sum(abs(imc.coef_W_).max(axis=1) > 0))
                d2_valid = int(sum(abs(imc.coef_H_).max(axis=1) > 0))

                results.append(
                    {
                        # "feature_id": ds["feature_id"],
                        "n_features": ds["n_features"],
                        "repeat_id": ds["repeat_id"],
                        "train_size": par_dtst["train_size"],
                        "n_splits": par_dtst["n_splits"],
                        "C_lasso": par_mdl["C_lasso"],
                        "C_group": par_mdl["C_group"],
                        "C_ridge": par_mdl["C_ridge"],
                        "rank": par_mdl["rank"],
                        "cv": cv,
                        "val_score": scores_valid,
                        "val_d1": d1_valid,
                        "val_d2": d2_valid,
                        "test_score": scores_test,
                        "test_d1": d1_test,
                        "test_d2": d2_test,
                    }
                )

df_results = pd.DataFrame(results)
df_results.head()

Datasets:   0%|          | 0/50 [00:00<?, ?it/s]/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: SparseEfficiencyWarning: Comparing a sparse matrix with 0 using == is inefficient, try using != instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/scipy/sparse/_index.py:143: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: SparseEfficiencyWarning: Comparing a sparse matrix with 0 using == is inefficient, try using != instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/scipy/sparse/_index.py:143: SparseEfficiencyWarning: Changing the spar

In [14]:
results

[{'n_features': 50,
  'repeat_id': 0,
  'train_size': 0.05,
  'n_splits': 3,
  'C_lasso': 1.0,
  'C_group': 1.0,
  'C_ridge': 1.0,
  'rank': 25,
  'cv': 0,
  'val_score': [0.639754137811794,
   0.6726108961889903,
   0.6780072904009721,
   0.6397768819724383,
   0.7544055944055944,
   0.5238948062965407,
   0.6156597169380612],
  'test_score': [0.7416467895741015,
   0.7750970148251781,
   0.744114034761249,
   0.7314296875,
   0.7832395756616576,
   0.6799152423538943,
   0.7087114337568058],
  'test_d1': 48,
  'test_d2': 50}]

In [ ]:
outfile = os.path.join(
    PATH_OUTPUT,
    f"results_sgimc_feature_{target_n_features:03d}.pkl",
)

with open(outfile, "wb") as f:
    pickle.dump(df_results, f)

print("Saved to:", outfile)
print(df_results.shape)
df_results.head()